## IR TrueFISP with the paper's 7-vial Resovist phantom

The Ehses et al. (MRM 2012) IR-TrueFISP sequence run on the **paper's actual 1.5 T phantom**:
7 vials of water + SPIO (Resovist) with anti-correlated T1 (shortens with dose) and T2
(lengthens with dose), laid out as 7 horizontal bands. Because the vials have T2 ~ T1 (unlike
brain, T2 << T1), the bSSFP steady state does **not** lock in within a few TRs, so the full
post-inversion recovery is visible: each band brightens toward its steady state on its own T1
timescale, giving strong temporal contrast. Same sequence / view-sharing / CG-SENSE pipeline as
the brain spin-sim demo; only the phantom differs.


In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from numpy import pi
import time
import pypulseq as pp
import torch
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm

sys.path.insert(0, "..")
from Modules import default_system

torch.set_default_device("cuda")

## Setup

In [ ]:
# Auxiliary functions


class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK, nTI = k.shape
        nC, nX, nY, TI = input_shape

        self.n_modes = (nX, nY)
        self.k = k
        self.forward_shape = (nC, nX, nY, 1, nTI)
        self.adjoint_shape = (nC, nK, nTI)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return y.view(torch.float32).ravel()


def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter), disable=disable_progressbar):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

## 1. The sequence

A single **180° inversion** (10 ms sinc, TI = 20 ms) followed by **700 TrueFISP spokes**
(TR = 4 ms, flip angle 60°, 64 samples each) on a golden-ratio (111.2°) radial trajectory.
TrueFISP **phase cycling** alternates the RF/ADC phase by 180° every TR so the readout samples
the *bright* transverse steady state.

In [ ]:
# Sequence definition


def truefisp_seq(
    FOV: float = 220e-3,
    N_x: int = 64,
    alpha: float = 60.0,
    slice_thickness: float = 6e-3,
    TRF: float = 0.8e-3,
    TBW: float = 2.0,
    BW: float = 890.0,
    TR: float = 4.0e-3,
    pre_dur: float = 0.6e-3,
    N_readout: int = 700,
    N_spokes: int = 1,
    R=1,
    N_dummy: int = 20,
    TI_start: float = 20e-3,
    TI_inter: float = 0.0,
    TD: float = 2000e-3,
    write_seq: bool = False,
    plot: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV
    _gr = seq.grad_raster_time

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.deg2rad(180),
        duration=10e-3,
        slice_thickness=slice_thickness,
        apodization=0.5,
        time_bw_product=4,
        system=system,
        return_gz=True,
        use="inversion",
    )

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=TRF,
        flip_angle=np.deg2rad(alpha),
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=TBW,
        return_gz=True,
        use="excitation",
    )

    # Readout duration = 1/BW, raster-aligned to both the grad raster (so the
    # gradient flat_time is valid) and N_x*adc_raster (so the ADC dwell is an
    # integer number of steps).
    _astep = N_x * seq.adc_raster_time
    _k = 1
    while abs(_k * _gr - round(_k * _gr / _astep) * _astep) > 1e-9:
        _k += 1
    readout_flat = int(round((1.0 / BW) / (_k * _gr))) * (_k * _gr)
    gx_base = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=readout_flat, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x,
        duration=gx_base.flat_time,
        delay=gx_base.rise_time,
        system=system,
    )

    pre_dur = int(np.ceil(pre_dur / _gr)) * _gr
    gx_pre_base = pp.make_trapezoid(
        channel="x", area=-gx_base.area / 2, duration=pre_dur, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=pre_dur, system=system
    )

    # TE is derived from the target TR so the whole TR raster-fits TR exactly.
    _fixed = (
        TRF
        + 2 * pre_dur
        + gx_base.flat_time
        + gx_base.rise_time
        + gx_base.fall_time
        + 2 * (gx_pre_base.rise_time + gx_pre_base.fall_time)
    )
    delay_TE = int(np.ceil(max(1e-4, TR - _fixed) / _gr)) * _gr
    TE = pre_dur + TRF / 2 + delay_TE + gx_base.flat_time / 2
    inv_duration = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil(
            (TI_start - inv_duration / 2 - pp.calc_duration(gz) / 2)
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    assert delay_TI_start > 0, "TI_start too short"

    gre_block_dur = (
        pp.calc_duration(gx_pre_base)
        + pp.calc_duration(gz)
        + delay_TE
        + pp.calc_duration(gx_base)  # readout
        + pp.calc_duration(gx_pre_base)  # in-plane rewind
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    GA_DEG = 111.246
    spoke_counter = 0
    tr_counter = 0

    for seg in range(N_spokes // R):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            # TrueFISP phase cycling: alternate the RF/ADC phase by 180 deg
            # every TR so the readout samples the bright transverse steady state.
            phase = (tr_counter % 2) * np.pi
            rf.phase_offset = phase
            adc.phase_offset = phase

            if is_dummy:
                angle = 0.0
            else:
                angle = np.deg2rad(spoke_counter * GA_DEG)
                spoke_counter += 1

            kx = np.cos(angle)
            ky = np.sin(angle)

            gx_pre = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_pre = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            gx_rew = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_rew = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            seq.add_block(rf, gz)
            seq.add_block(gx_pre, gy_pre, gz_reph)
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(
                channel="x",
                amplitude=gx_base.amplitude * kx,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )
            gy = pp.make_trapezoid(
                channel="y",
                amplitude=gx_base.amplitude * ky,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )

            if is_dummy:
                seq.add_block(gx, gy)
            else:
                seq.add_block(gx, gy, adc)

            seq.add_block(gx_rew, gy_rew, gz_reph)

            if delay_TI_inter > 0 and s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))
            tr_counter += 1

        seq.add_block(pp.make_delay(TD))

    ok, err = seq.check_timing()
    if ok:
        print("2-D TrueFISP timing check passed")
        print(f"  Target TR          : {TR * 1000:.3f} ms")
        print(f"  GRE block dur (TR) : {gre_block_dur * 1000:.3f} ms")
        print(f"  TE                 : {TE * 1000:.3f} ms")
        print(f"  Readout BW         : {N_x / gx_base.flat_time / 1e3:.1f} kHz/px")
        print(f"  Readout train      : {N_readout * gre_block_dur * 1000:.0f} ms")
        est = N_spokes * (
            inv_duration + delay_TI_start + N_readout * gre_block_dur + TD
        )
        print(f"  Scan time est       : {est / 60:.1f} min")
    else:
        print(f"Timing FAILED: {err}")

    seq.set_definition("FOV", [FOV, FOV, slice_thickness])
    seq.set_definition("Name", "us_trueFISP_2D_radial")
    if write_seq:
        seq.write("us_trueFISP_2D_Radial.seq")
    if plot:
        seq.plot(time_range=(0.2, 0.225))
    return seq, gre_block_dur, TI_start


In [ ]:
# Parameters (Ehses et al. in-vivo, 1.5 T)
device = torch.device("cuda")
N_x = 64
N_channels = 8
FOV = 220e-3
slice_thickness = 6e-3
alpha = 60.0        # excitation flip angle
TRF = 0.8e-3        # RF pulse duration (TRF/TR = 20%)
TBW = 2.0           # sinc time-bandwidth product
BW = 890.0          # readout bandwidth (Hz/px)
TR = 4.0e-3         # repetition time
pre_dur = 0.6e-3    # pre-scan / rewind gradient duration
N_readout = 700
N_spokes = 1
R = 1
N_dummy = 0         # sample right after the inversion
TI_start = 20e-3
TI_inter = 0.0
TD = 2000e-3

seq, gre_block_dur, TI_start = truefisp_seq(
    FOV=FOV, N_x=N_x, slice_thickness=slice_thickness, alpha=alpha,
    TRF=TRF, TBW=TBW, BW=BW, TR=TR, pre_dur=pre_dur,
    write_seq=True, plot=False,
)


## 2. Phantom: 7 Resovist vials

`load_phantom` provides the grid + coil wiring; we overwrite T1/T2/PD with the 7 vial bands
(SPIO dose shortens T1, lengthens T2). `B0=0` (on-resonant), `B1=1`.


In [ ]:
# The paper's 1.5 T phantom: 7 vials of water + SPIO (Resovist), as 7 horizontal bands.
device = torch.device("cuda")
phantom = mr0.util.load_phantom(size=(N_x, N_x))

coil_sensitivities_recon = cyclic_head_coil((N_channels, N_x, N_x, 1)).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))[..., 0]
coil_sensitivities_sim = cyclic_head_coil(
    (N_channels, N_x, N_x, 1),
    fov_scaling=(phantom.size[0] / FOV, phantom.size[1] / FOV, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)

# 7 Resovist bands (row bands). SPIO dose shortens T1 and lengthens T2 (anti-correlated).
vial_T1 = torch.tensor([800, 700, 600, 500, 400, 300, 200]) / 1000.0   # ms, long -> short
vial_T2 = torch.tensor([120, 160, 200, 250, 300, 350, 400]) / 1000.0   # ms, short -> long
T1_map = torch.zeros(phantom.T1.shape); T2_map = torch.zeros(phantom.T2.shape); PD_map = torch.zeros(phantom.PD.shape)
band = N_x // len(vial_T1)
for i in range(len(vial_T1)):
    r0 = i * band
    r1 = N_x if i == len(vial_T1) - 1 else (i + 1) * band
    T1_map[r0:r1] = vial_T1[i]; T2_map[r0:r1] = vial_T2[i]; PD_map[r0:r1] = 1.0
phantom.T1 = T1_map; phantom.T2 = T2_map; phantom.PD = PD_map
print(f"7 Resovist vial bands: T1 {vial_T1.min()*1000:.0f}-{vial_T1.max()*1000:.0f} ms, "
      f"T2 {vial_T2.min()*1000:.0f}-{vial_T2.max()*1000:.0f} ms (row bands, long-T1 at top)")
data = phantom.build()


## 3. Simulation (ground-truth time-domain spin simulation)

In [ ]:
# Simulate the sequence (ground-truth time-domain spin simulation)
mr0_seq = mr0.Sequence.import_file("us_trueFISP_2D_Radial.seq")
torch.manual_seed(0)
signal = mr0.isochromat_sim(
    mr0_seq, data.cuda(), spin_count=256, spin_dist="rand", print_progress=True,
)
signal = signal.permute((1, 0)).contiguous()   # (N_channels, N_samples)
print(f"simulated signal: {tuple(signal.shape)}")


## 3. Reconstruction

The 700 spokes are grouped into time frames by the paper's **Fibonacci "view-sharing" temporal
k-space filter** (Ehses et al., Fig. 1b): each frame takes its k-space *center* from the nearest
projections and progressively *more* from farther in time toward the periphery (a "butterfly"
mask), with Ram-Lak 1/r density compensation. Each frame is reconstructed with **CG-SENSE**
(adjoint regridding + a conjugate-gradient SENSE solve).

In [ ]:
# Extract k-space trajectory

kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

In [ ]:
# Fibonacci "view-sharing" temporal k-space filter (Ehses et al., Fig. 1b).
#
# A frame centered on projection n takes its k-space CENTER from only the N_center
# projections closest in time to n (fast contrast update), and progressively MORE
# projections from farther in time toward the periphery. The per-radius projection
# count jumps to the next Fibonacci number as the Nyquist rule (count ~ radius)
# demands, producing the "butterfly" mask. Frames advance by `frame_advance`.
# A Ram-Lak 1/r density-compensation is applied to the data.
#
# Replaces the plain consecutive-spoke block grouping: the result keeps the same
# downstream variable names (data_reshaped, k_traj_adc_reshaped, N_readout_effective)
# so the NUFFT / CG-SENSE / plot cells work unchanged.

N_center = 8          # projections for the k-space center
N_max = 55            # outermost Fibonacci ring; also the frame time window
frame_advance = 4     # filter advance per frame (paper: 8 -> 32 ms; 4 -> 16 ms
                      #   for finer temporal resolution of the IR evolution)
use_dcf = False       # NOT needed: iterative recon models the exact sampling (gridding-only correction)

FIB = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
def _next_fib(x):
    return next(f for f in FIB if f >= x)

n_samples_total = k_traj_adc.shape[1]
assert n_samples_total % N_x == 0, "trajectory is not a whole number of spokes"
n_spokes_total = n_samples_total // N_x
assert n_spokes_total > N_max, "need more spokes than N_max for view-sharing"

# radius of each radial sample (0 at center -> 1 at the edge) and its ring count
r = np.linspace(0.0, 1.0, N_x)
ring = np.array([_next_fib(max(N_center, int(np.ceil(N_max * ri)))) for ri in r])
# Ram-Lak 1/r DCF (center clamped to the first non-zero radius)
dcf = np.where(r > 0, 1.0 / r, 1.0 / r[1])

# butterfly offsets: for sample i, the ring[i] projections centered on the frame
djs, is_ = [], []
for i in range(N_x):
    nproj = int(ring[i])
    lo = -(nproj // 2)
    djs.extend(range(lo, lo + nproj))
    is_.extend([i] * nproj)
dj = np.asarray(djs, dtype=np.int64)        # (K,) relative projection offset
ii = np.asarray(is_, dtype=np.int64)        # (K,) radial sample index
K = dj.size
w = dcf[ii] if use_dcf else np.ones(K)      # (K,) DCF per butterfly point

# valid frames: the outermost window [n - N_max//2, n + N_max//2] must fit
n0 = N_max // 2
n_frames = (n_spokes_total - N_max - 1) // frame_advance + 1
centers = n0 + frame_advance * np.arange(n_frames)

# per-frame flat k-space sample indices for the butterfly (n_frames, K)
idx = (centers[:, None] + dj[None, :]) * N_x + ii[None, :]

# Butterfly gather on the CPU (robust for the complex signal), then move the
# result back to the accelerator. flat index is frame-major (frame, point).
sig_cpu = signal.cpu().numpy()          # (C, S) complex64
k_cpu = k_traj_adc.cpu().numpy()        # (2, S) float32
flat = idx.reshape(-1)                  # (n_frames * K,)
k_traj_adc_reshaped = (
    torch.from_numpy(k_cpu[:, flat].reshape(2, n_frames, K).transpose(0, 2, 1).copy())
    .float().to(k_traj_adc.device)
)                                                            # (2, K, n_frames)
data_reshaped = (
    torch.from_numpy(
        (sig_cpu[:, flat].reshape(N_channels, n_frames, K).transpose(0, 2, 1).copy()
         * w[None, :, None]).astype(np.complex64)
    ).to(signal.device)
)                                                            # (C, K, n_frames)

N_readout_effective = n_frames
coil_sensitivities_recon_tiled = torch.tile(
    coil_sensitivities_recon[:, :, :, None], (1, 1, 1, N_readout_effective)
)
print(f"view-sharing: {n_spokes_total} spokes -> {n_frames} frames, "
      f"K={K} pts/frame, rings={list(np.unique(ring))}, "
      f"N_center={N_center} N_max={N_max} advance={frame_advance} dcf={use_dcf}")


In [ ]:
# CG-SENSE reconstruction:  A = F @ C  (non-uniform Fourier @ coil combine)
F = NonuniformFourierTransformOperator(
    k_traj_adc_reshaped, (N_channels, N_x, N_x, N_readout_effective)
)
C = ChannelOperator(coil_sensitivities_recon_tiled, (N_x, N_x, N_readout_effective))
A = F @ C
AHd = A.H @ data_reshaped.contiguous().view(torch.float32).ravel()
AHd_image = AHd.view(torch.complex64).reshape((N_x, N_x, N_readout_effective))

Q = A.H @ A
x = torch.zeros(N_x * N_x * N_readout_effective, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, AHd, x, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((N_x, N_x, N_readout_effective))
print(f"reconstructed {N_readout_effective} frames, shape {tuple(x_recon_image.shape)}")

torch.save(x_recon_image.cpu(), '/tmp/opencode/recon_vial_complex.pt')


## 5. Results - the per-vial recovery

Each band has its own T1, so it recovers from the inversion on its own timescale. The **magnitude**
of each band dips after the 180 deg pulse (the band inverts) and rises back to its (bright)
steady state; the higher-T2 / longer-T1 bands dominate the late frames. Shown as per-band mean
curves and a grayscale image series (shared window so the temporal contrast is visible).


In [ ]:
# Per-vial (per-row-band) inversion-recovery.
ximg = x_recon_image.cpu().numpy()                       # (N_x, N_x, Nf) complex
Nf = ximg.shape[2]; n0 = N_max // 2
t_frames = (n0 + frame_advance * np.arange(Nf)) * TR * 1e3   # ms
phase = np.angle(ximg[:, :, -1])
signed = np.real(ximg * np.exp(-1j * phase)[:, :, None])
mag = np.abs(ximg)
print(f"frames: {Nf}, advance {frame_advance} TR -> {frame_advance*TR*1e3:.0f} ms; "
      f"span {t_frames[0]:.0f}-{t_frames[-1]:.0f} ms")

band = N_x // 7
def band_mask(i):
    m = np.zeros((N_x, N_x), bool)
    r0 = i * band; r1 = N_x if i == 6 else (i + 1) * band
    m[r0:r1, :] = True
    return m
mag_flat = mag.reshape(-1, Nf); signed_flat = signed.reshape(-1, Nf)

plt.figure(figsize=(8, 5))
for i in range(7):
    m = band_mask(i); t1 = vial_T1[i].item()*1000; t2 = vial_T2[i].item()*1000
    plt.plot(t_frames, mag_flat[m.reshape(-1)].mean(0), lw=2,
             label=f"band {i+1}: T1={t1:.0f}/T2={t2:.0f} ms")
plt.axhline(0, color="k", lw=0.8); plt.xlabel("time (ms)"); plt.ylabel("mean magnitude")
plt.title("Per-vial IR recovery (7 Resovist bands) - each band recovers on its own T1")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

# grayscale image series over the early recovery (5-frame average per displayed frame)
vmax = float(np.percentile(mag, 95))
def den(i):
    lo, hi = max(0, i - 2), min(Nf, i + 3); return mag[:, :, lo:hi].mean(-1)
f_end = min(Nf - 1, int((1200.0 - t_frames[0]) / (frame_advance * TR * 1e3)))
frame_idx = np.sort(np.unique(np.linspace(0, f_end, 10).round().astype(int)))
nrows, ncols = 3, (len(frame_idx) + 2) // 3
plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
plt.suptitle("Per-vial IR magnitude series (grayscale) - bands recover on their own T1")
for j, i in enumerate(frame_idx):
    ax = plt.subplot(nrows, ncols, j + 1)
    plt.imshow(den(i).T, cmap="gray", vmin=0, vmax=vmax)
    plt.title(f"{t_frames[i]:.0f} ms", fontsize=9); plt.axis("off")
for j in range(len(frame_idx), nrows * ncols):
    plt.subplot(nrows, ncols, j + 1); plt.axis("off")
plt.show()
